<a href="https://colab.research.google.com/github/SandeepD16/DatasheetRAG/blob/main/DatasheetGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y transformers sentence-transformers langchain langchain-community langchain-core

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: sentence-transformers 5.3.0
Uninstalling sentence-transformers-5.3.0:
  Successfully uninstalled sentence-transformers-5.3.0
Found existing installation: langchain 0.1.16
Uninstalling langchain-0.1.16:
  Successfully uninstalled langchain-0.1.16
Found existing installation: langchain-community 0.0.32
Uninstalling langchain-community-0.0.32:
  Successfully uninstalled langchain-community-0.0.32
Found existing installation: langchain-core 0.1.53
Uninstalling langchain-core-0.1.53:
  Successfully uninstalled langchain-core-0.1.53


In [ ]:
from langchain.prompts import PromptTemplate

In [ ]:
!pip install -q \
transformers==4.41.2 \
sentence-transformers==2.7.0 \
langchain==0.1.16 \
langchain-community==0.0.32 \
faiss-cpu \
pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain.chains import RetrievalQA

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

In [ ]:
loader = PyPDFLoader("Datasheet_ESP32.PDF")
documents = loader.load()

print("Total pages:", len(documents))

Total pages: 60


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

docs = text_splitter.split_documents(documents)

print("Total chunks:", len(docs))

Total chunks: 166


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings loaded ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embeddings loaded ✅


In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text2text-generation",   # ✅ CORRECT
    model="google/flan-t5-large",
    max_length=256
)
from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
prompt_template = """
Use the following context from a microcontroller datasheet to answer the question.
DO NOT copy the text directly.

IMPORTANT:
- Do NOT give short answers
- Always explain in detail. For exaple if I am asking about how many interrupts are available. Always start your
answer with explaining what is an interrupt.




If the answer is not present in the context, say:
"Not available in datasheet"

Context:
{context}

Question:
{question}

Answer:
"""
retriever = vectorstore.as_retriever()
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)



qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT}
)

In [ ]:
query = "What is SRAM specification? Explain"

raw_result = qa.run(query)

print(result)

explanation = llm.invoke(
    f"Explain this in simple terms for an engineering student, expalining each technical term in detail:\n{raw_result}"
)

print(explanation)

520 KB of on-chip SRAM for data and instructions
The chip has 520 KB of on-chip SRAM for data and instructions.


In [ ]:
query = "What is ADC resolution? Explain in detail, what it means and why it is important."

result = qa.run(query)

print(result)

Not available in datasheet
